In [ ]:
### CONSTANTS AND HELPERS
import sqlite3, io, pandas as pd, base64, matplotlib.pyplot as plt, sys, os, glob, pytz, IPython.core.display as ip, plotly.express as px
from IPython.display import display, HTML
from datetime import datetime

surveyYear = '2025'
aboveDamOnly = False
use_v2_report = True

additionalFilterForAboveDam = "AND CAST(Distance AS int) > 310" if aboveDamOnly else ""

surveyURIs = {'2019':'https://five.epicollect.net/api/export/entries/salmon-survey-2019?form_ref=397fba6ecc674b74836efc190840c42d_5d6f454667a28&per_page=100',
              '2020':'https://five.epicollect.net/api/export/entries/salmon-survey-2020?form_ref=f550ab6c4dab44f49bcc33b7c1904be9_5d6f454667a28&per_page=100',
              '2021':'https://five.epicollect.net/api/export/entries/salmon-survey-2021?form_ref=ad5ffedf0a3246a18934e6ec36ed9569_5d6f454667a28&per_page=100',
              '2022':'https://five.epicollect.net/api/export/entries/salmon-survey-2022?form_ref=d46b5d8451f8410ea407bae5c8eb9f49_5d6f454667a28&per_page=100'}
salmonURIs = {'2019':'https://five.epicollect.net/api/export/entries/salmon-survey-2019?form_ref=397fba6ecc674b74836efc190840c42d_5d6f509867795&per_page=500',
              '2020':'https://five.epicollect.net/api/export/entries/salmon-survey-2020?form_ref=f550ab6c4dab44f49bcc33b7c1904be9_5d6f509867795&per_page=500',
              '2021':'https://five.epicollect.net/api/export/entries/salmon-survey-2021?form_ref=ad5ffedf0a3246a18934e6ec36ed9569_5d6f509867795&per_page=500',
              '2022':'https://five.epicollect.net/api/export/entries/salmon-survey-2022?form_ref=d46b5d8451f8410ea407bae5c8eb9f49_5d6f509867795&per_page=500',
              '2023':'https://kf.kobotoolbox.org/api/v2/assets/a6dEG7tnrtwjrmituAdL5k/data/?format=json',
              '2024':'https://kf.kobotoolbox.org/api/v2/assets/ae8BCoHi4EmwnzP2ShmSUw/data/?format=json',
              '2025':'https://kf.kobotoolbox.org/api/v2/assets/a5WFFCGawCP3aTLHdRjrca/data/?format=json'
             }

plotly_font_family = {'family': "Arial, Helvetica, sans-serif"}
plotly_year_menu_attrs = {
    'type': 'buttons',
    'direction': 'right',
    'showactive': True,
    'xanchor': 'left',
    'yanchor': 'top',
    'bgcolor': 'lightgrey',
    'bordercolor': 'grey',
    'borderwidth': 1
}

IN_COLAB = 'google.colab' in sys.modules

In [ ]:
### DB AND TABLE SETUP

def getFigureAsHTML():
    IObytes = io.BytesIO()
    plt.savefig(IObytes, format = 'png')
    IObytes.seek(0)
    encodedPlot = base64.b64encode(IObytes.read()).decode("utf-8")
    return '<img src=\'data:image/png;base64,{}\'>'.format(encodedPlot)

#for running locally
def clearPreviousReports():
    for fileName in glob.glob('*salmonReport.html'):
        print(f"Previous report file exists. Deleting {fileName}")
        os.remove(fileName)

def getSurveyStats(year):
    dead_to_date_query = f'''
    WITH salmon_counts AS (
        SELECT
            Survey_Date,
            COALESCE(SUM(CASE WHEN Species in ('Chum', 'Coho', 'Unknown', 'Sea-run Cutthroat', 'Sea-run_Cutthroat') AND Type in ('Dead', 'Remnant') THEN Quantity END), 0) AS total_dead_salmon_count,
            COALESCE(SUM(CASE WHEN Species in ('Chum', 'Coho', 'Unknown', 'Sea-run_Cutthroat', 'Sea-run Cutthroat') AND Type = 'Live' THEN Quantity END), 0) AS total_live_salmon_count,
            COALESCE(SUM(CASE WHEN Species in ('Chum', 'Coho', 'Unknown', 'Sea-run_Cutthroat', 'Sea-run Cutthroat') AND Type in ('Live', 'Dead', 'Remnant') THEN Quantity END), 0) AS total_salmon_count,
            COALESCE(SUM(CASE WHEN Species = 'Chum' AND Type in ('Dead', 'Remnant') THEN Quantity END), 0) AS dead_chum_count,
            COALESCE(SUM(CASE WHEN Species = 'Chum' AND Type = 'Live' THEN Quantity END), 0) AS live_chum_count,
            COALESCE(SUM(CASE WHEN Species = 'Coho' AND Type in ('Dead', 'Remnant') THEN Quantity END), 0) AS dead_coho_count,
            COALESCE(SUM(CASE WHEN Species = 'Coho' AND Type = 'Live' THEN Quantity END), 0) AS live_coho_count,
            COALESCE(SUM(CASE WHEN Species in ('Resident_Cutthroat', 'Sea-run_Cutthroat', 'Resident Cutthroat', 'Sea-run Cutthroat', 'Cutthroat') AND Type in ('Dead', 'Remnant') THEN Quantity END), 0) as dead_cutthroat_count,
            COALESCE(SUM(CASE WHEN Species in ('Resident_Cutthroat', 'Sea-run_Cutthroat', 'Resident Cutthroat', 'Sea-run Cutthroat', 'Cutthroat') AND Type = 'Live' THEN Quantity END), 0) as live_cutthroat_count,
            COALESCE(SUM(CASE WHEN Species = 'Unknown' AND Type in ('Dead', 'Remnant') THEN quantity END), 0) AS dead_unknown_count,
            COALESCE(SUM(CASE WHEN Species = 'Unknown' AND Type = 'Live' THEN quantity END), 0) AS live_unknown_count,
            COALESCE(SUM(CASE WHEN Type = 'Redd' THEN Quantity END), 0) as redd_count
        FROM
            salmon
        WHERE
            year = {year} {additionalFilterForAboveDam}
        GROUP BY
            Survey_Date
    ), running_counts AS (
        SELECT
            Survey_Date,
            SUM(dead_chum_count) OVER (ORDER BY Survey_Date) AS running_total_dead_chum,
            SUM(dead_chum_count) OVER (ORDER BY Survey_Date) + live_chum_count AS running_total_all_chum,
            SUM(dead_coho_count) OVER (ORDER BY Survey_Date) AS running_total_dead_coho,
            SUM(dead_coho_count) OVER (ORDER BY Survey_Date) + live_coho_count AS running_total_all_coho,
            SUM(dead_cutthroat_count) OVER (ORDER BY Survey_Date) AS running_total_dead_cutthroat,
            SUM(dead_cutthroat_count) OVER (ORDER BY Survey_Date) + live_cutthroat_count AS running_total_all_cutthroat,
            SUM(dead_unknown_count) OVER (ORDER BY Survey_Date) AS running_total_dead_unknown,
            SUM(dead_unknown_count) OVER (ORDER BY Survey_Date) + live_unknown_count AS running_total_all_unknown,
            SUM(total_dead_salmon_count) OVER (ORDER BY Survey_Date) AS running_total_dead_salmon,
            SUM(total_dead_salmon_count) OVER (ORDER BY Survey_Date) + total_live_salmon_count AS running_total_all_salmon
        FROM
            salmon_counts
    )
    SELECT        
        sc.Survey_Date,
        sc.total_dead_salmon_count,
        sc.total_live_salmon_count,
        sc.total_salmon_count,
        sc.dead_chum_count,
        sc.live_chum_count,
        sc.dead_coho_count,
        sc.live_coho_count,
        sc.dead_cutthroat_count,
        sc.live_cutthroat_count,
        sc.dead_unknown_count,
        sc.live_unknown_count,
        sc.redd_count,
        rc.running_total_dead_chum,
        rc.running_total_all_chum,
        rc.running_total_dead_coho,
        rc.running_total_all_coho,
        rc.running_total_dead_cutthroat,
        rc.running_total_all_cutthroat,
        rc.running_total_dead_unknown,
        rc.running_total_all_unknown,
        rc.running_total_dead_salmon,
        rc.running_total_all_salmon
    FROM
        salmon_counts sc
    JOIN running_counts rc ON sc.Survey_Date = rc.Survey_Date;
    '''
    return pd.read_sql(dead_to_date_query, connection)

def getScatterMap(df, figureTitle):
    fig = px.scatter_mapbox(df, lat='Latitude', lon='Longitude', color='Type', labels={'Type':'Type'}, color_discrete_map={'Live': 'teal', 'Redd': 'red', 'Dead': 'black'},
                    center=dict(lat=47.71157, lon=-122.3759), zoom=15, hover_name = 'Type', hover_data = ['Distance', 'Quantity', 'Species', 'Sex', 'Accuracy'],
                    mapbox_style='open-street-map', title=figureTitle)
    fig.layout.coloraxis.showscale = False
    fig.update_layout(title_x=0.5)
    fig.show()
    return fig.to_html(include_plotlyjs="cdn")
            
create_salmon_table_query = '''
    CREATE TABLE IF NOT EXISTS salmon (
        _id STRING PRIMARY KEY,
        Survey_Date DATE,
        year DATE,
        Quantity INTEGER,
        Distance INTEGER,
        Stream TEXT,
        Type TEXT,
        Species TEXT,
        Predation TEXT,
        Length FLOAT,
        Width FLOAT,
        Spawned TEXT,
        Sex TEXT,
        Latitude FLOAT,
        Longitude FLOAT,
        Accuracy FLOAT
    );
'''
def getAllFiles():
    !git clone https://github.com/slfisco/Survey-Notebook.git

if IN_COLAB:
    getAllFiles()
else:
    clearPreviousReports()
reddsTable = yearByYearCountPlot = countPlot = surveyStatsTable = yearScatterMap = latestScatterMap = None
maxSurveyChum = maxSurveyChumDate = maxSurveyCoho = maxSurveyCohoDate = chumSpawnSuccess = cohoSpawnSuccess = chumFemaleSpawnSuccess = chumMaleSpawnSuccess = chumPredation = cohoPredation = None   
connection = sqlite3.connect(":memory:")
cursor = connection.cursor()
cursor.execute(create_salmon_table_query)

In [ ]:
### DATA LOADING
import time
import requests

salmon_insert_query = '''
        INSERT OR IGNORE INTO salmon (
        _id,
        Survey_Date,
        year,
        Quantity,
        Distance,
        Stream,
        Type,
        Species,
        Predation,
        Length,
        Width,
        Spawned,
        Sex,
        Latitude,
        Longitude,
        Accuracy
        ) VALUES (?, ?, ?, COALESCE(?,1), ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    '''

def getDataPage(uri, label):
    is_epicollect = "epicollect" in uri
    if is_epicollect:
        delays = [5, 30, 60, 120]
    else:
        delays=[1, 10, 30]
    for attempt, delay in enumerate(delays, start=1):
        time.sleep(delay)
        try:
            response = requests.get(uri)
            response.raise_for_status()
            data = response.json()
        except requests.RequestException as exc:
            print(f"[{label}] fetch error attempt {attempt} for {uri}: {exc}")
        except ValueError as exc:
            print(f"[{label}] invalid JSON attempt {attempt} for {uri}: {exc}")
        else:
            if is_epicollect:
                entries = data.get("data", {}).get("entries")
                next_uri = data.get("links", {}).get("next")
            else:
                entries = data.get("results")
                next_uri = data.get("next")

            if isinstance(entries, list):
                return entries, next_uri

            print(f"[{label}] malformed response attempt {attempt} for {uri}: missing entries")

        if attempt == len(delays):
            print(f"[{label}] giving up after {attempt} attempts for {uri}")
            return None, None

        next_delay = delays[attempt] if attempt < len(delays) else 0
        print(f"[{label}] retrying in {next_delay} seconds...")
    return None, None

## for epicollect data to associate salmon to a survey date
def getSurveyDates(uri):
    surveyDates = {}
    entries, _ = getDataPage(uri, label="survey-dates")
    if not entries:
        return surveyDates
    for entry in entries:
        surveyDate = datetime.strptime(entry['Survey_Date'], "%m/%d/%Y").strftime("%Y-%m-%d")
        surveyDates[entry['ec5_uuid']] = surveyDate
    return surveyDates
    
def getLocation(entry, isEpicollect):
    latitude = longitude = accuracy = location = None
    if isEpicollect:
        latitude = entry.get('Location').get('latitude')
        longitude = entry.get('Location').get('longitude')
        accuracy = entry.get('Location').get('accuracy')
    else:
        location = entry.get('Location')
        if location is not None: 
            location = location.split()
            latitude = location[0]
            longitude = location[1]
            accuracy = location[3]
    return latitude, longitude, accuracy
        
def processEntries(entries, isEpicollect, year, surveyDates):
    for entry in entries:
        location = getLocation(entry, isEpicollect)
        latitude = location[0]
        longitude = location[1]
        accuracy = location[2]
        values = (
            entry.get('ec5_uuid') if isEpicollect else entry.get('_id'),
            surveyDates[entry.get('ec5_parent_uuid')] if isEpicollect else entry.get('Survey_Date'),
            year,
            entry.get('Quantity', 1),
            entry.get('Distance'),
            entry.get('Stream'),
            entry.get('Type'),
            entry.get('Species'),
            entry.get('Predation'),
            entry.get('Length_Inches') if isEpicollect else entry.get("Length"),
            entry.get('Width_Inches') if isEpicollect else entry.get("Width"),
            entry.get('Spawning_Success') if isEpicollect else entry.get("Spawned"),
            entry.get('Sex'),
            latitude,
            longitude,
            accuracy
        )
        cursor.execute(salmon_insert_query, values)

def loadSurveyYear(year):
    print(f'loading for year: {year}')
    uri = salmonURIs[year]
    isEpicollect = "epicollect" in uri
    surveyDates = getSurveyDates(surveyURIs[year]) if isEpicollect else None
    while uri:
        entries, uri = getDataPage(uri, label=year)
        if entries is None:
            print(f"Stopping load for year {year} because the page failed or was malformed.")
            break
        processEntries(entries, isEpicollect, year, surveyDates)
        
print('loading salmon into database')
for year in salmonURIs:
    loadSurveyYear(year)

In [ ]:
def getMaxSurveyTotal(df, columnName):
    max_row = df[columnName].values.argmax()
    total = df.iloc[max_row][columnName]
    return total

def getMaxSurveyDate(df, columnName):
    max_row = df[columnName].values.argmax()
    calcDate = df.iloc[max_row]["Survey_Date"]
    return calcDate
    
df = getSurveyStats(surveyYear)
maxSurveyChum = getMaxSurveyTotal(df, 'running_total_all_chum')
maxSurveyChumDate = getMaxSurveyDate(df, 'running_total_all_chum')
maxSurveyCoho = getMaxSurveyTotal(df, 'running_total_all_coho')
maxSurveyCohoDate = getMaxSurveyDate(df, 'running_total_all_coho')
print(f'max survey chum: {maxSurveyChum}')
print(f'max survey chum date: {maxSurveyChumDate}')
print(f'max survey coho: {maxSurveyCoho}')
print(f'max survey coho date: {maxSurveyCohoDate}')

In [ ]:
def displaySurveyStatsTable():
    tableDf = getSurveyStats(surveyYear)[['Survey_Date', 'live_chum_count', 'dead_chum_count', 'live_coho_count', 'dead_coho_count', 'live_cutthroat_count', 'dead_cutthroat_count', 'live_unknown_count', 'dead_unknown_count']]
    table = tableDf.rename(columns={'Survey_Date': 'Survey Date', 'live_chum_count': 'Live Chum', 'dead_chum_count': 'Dead Chum', 'live_coho_count': 'Live Coho', 'dead_coho_count': 'Dead Coho', 'live_cutthroat_count': 'Live Cutthroat', 'dead_cutthroat_count': 'Dead Cutthroat', 'live_unknown_count': 'Live Unknown', 'dead_unknown_count': 'Dead Unknown'}).style.hide().to_html()
    display(ip.HTML(table))
    return table
surveyStatsTable = displaySurveyStatsTable()

In [ ]:
### DAILY FISH COUNT BY TYPE AND SPECIES
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from IPython.display import display, HTML
import plotly.graph_objects as go

def displayCountPlot(df):
    df['Survey_Date'] = pd.to_datetime(df['Survey_Date']).dt.strftime('%m/%d')
    df = df.rename(columns={'live_chum_count': 'Live Chum', 'dead_chum_count': 'Dead Chum', 'live_coho_count': 'Live Coho', 'dead_coho_count': 'Dead Coho'})
    ax = df.plot(kind='barh', stacked=True, x='Survey_Date', y=['Live Chum', 'Dead Chum', 'Live Coho', 'Dead Coho'], xlabel='Survey Date', ylabel='Count', title=f'{surveyYear} Fish Count')
    ax.invert_yaxis()
    return getFigureAsHTML()

def displayCountPlotChartByYear(years=list(salmonURIs.keys()), default_year=surveyYear):
    categories = ['Live Chum', 'Dead Chum', 'Live Coho', 'Dead Coho']
    color_map = {
        'Live Chum': 'skyblue',
        'Dead Chum': 'orange',
        'Live Coho': 'lime',
        'Dead Coho': 'red'
    }
    title_position = {
        'x': 0.5,
        'y': 0.96
    }

    traces = []
    for year in years:
        df = getSurveyStats(year)
        df['Survey_Date'] = pd.to_datetime(df['Survey_Date']).dt.strftime('%m/%d')
        df = df.rename(columns={
            'live_chum_count': 'Live Chum',
            'dead_chum_count': 'Dead Chum',
            'live_coho_count': 'Live Coho',
            'dead_coho_count': 'Dead Coho'
        })
        long_df = df.melt(
            id_vars=['Survey_Date'],
            value_vars=categories,
            var_name='Category',
            value_name='Count'
        )

        for category in categories:
            subset = long_df[long_df['Category'] == category]
            traces.append(go.Bar(
                name=category,
                x=subset['Count'],
                y=subset['Survey_Date'],
                orientation='h',
                marker_color=color_map[category],
                visible=(year == default_year),
                hovertemplate='<b>%{y}</b><br>%{fullData.name}: %{x}<extra></extra>'
            ))

    buttons = []
    for year in years:
        visible = [(y == year) for y in years for _ in categories]
        visible += [(y == year) for y in years]
        buttons.append(dict(
            label=year,
            method='update',
            args=[
                {'visible': visible},
                {
                    'title': {
                        'text': f'{year} Fish Count',
                        **title_position
                    }
                }
            ]
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        barmode='stack',
        font=plotly_font_family,
        yaxis={'autorange': 'reversed', 'title': 'Survey Date'},
        title={
            'text': f'{default_year} Fish Count',
            **title_position
        },
        title_x=0.5,
        legend_title_text='Count Type',
        updatemenus=[{**dict(
            active=years.index(default_year),
            buttons=buttons,
            pad={'r': 10, 't': 10},
            x=0,
            y=1.15,
        ), **plotly_year_menu_attrs}],
        margin=dict(t=90, b=20, l=60, r=20),
        height=480,
    )

    return fig.to_html(include_plotlyjs='cdn', full_html=False)

if use_v2_report:
    countPlotByYear = displayCountPlotChartByYear()
    display(HTML(countPlotByYear))
else:
    countPlot = displayCountPlot(getSurveyStats(surveyYear))


In [ ]:
### REDDS TABLE. USED TO HELP SURVEY TEAM AVOID REDDS
redds_table_query = f'''
SELECT
    Stream, Distance, Survey_Date
FROM
    salmon
WHERE Type = 'Redd' AND year = {surveyYear}
'''
def createReddsTable():
    table = pd.read_sql(redds_table_query, connection).style.hide().to_html()
    display(ip.HTML(table))
    return table
reddsTable = createReddsTable()

In [ ]:
def plotBarH(query, title, colors):
    df = pd.read_sql(query, connection)
    if df.isna().all().all():
        print(f'No values found for {title}. Skipping.')
        return None
    df = df.loc[:, (df != 0).all(axis=0)]
    ax = df.plot(kind='barh', stacked=True, title=title, color=colors)
    for container in ax.containers:
        ax.bar_label(container, label_type='center', fmt='%.f%%')
    ax.yaxis.set_tick_params(labelleft=False, left=False)
    ax.xaxis.set_tick_params(labelleft=False, left=False)
    return getFigureAsHTML()

In [ ]:
### SPAWN SUCCESS
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plotSpawning(species, sex=''):
    sexFilter = f"AND Sex = '{sex}'" if sex != '' else ''
    colors = {'Partially Spawned':'yellow', 'Spawned':'green', 'Unknown':'gray', 'Unspawned':'red'}
    query = f'''
    SELECT
        100 * CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' AND Spawned = 'Spawned' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' THEN _id END) AS float) AS Spawned,
        100 * CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' AND Spawned = 'Unspawned' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' THEN _id END) AS float) AS Unspawned,
        100 * CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' AND Spawned in ('Partially_spawned', 'Partially spawned') THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' THEN _id END) AS float) AS [Partially Spawned],
        100 * CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' AND Spawned = 'Unknown' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' {sexFilter} AND Type = 'Dead' THEN _id END) AS float) AS Unknown
    FROM
        salmon
    WHERE year = {surveyYear}
    '''
    return plotBarH(query, f'{species} {sex}{" " if sex else ""}Spawning Success', colors)

def displaySpawningChartsByYear(years=list(salmonURIs.keys()), default_year=surveyYear):
    species_configs = [
        ('Chum', '', 1, 1),
        ('Coho', '', 1, 2),
        ('Chum', 'Female', 2, 1),
        ('Chum', 'Male', 2, 2),
    ]
    colors = {
        'Spawned': 'green',
        'Unspawned': 'red',
        'Partially Spawned': 'gold',
        'Unknown': 'lightgray'
    }
    categories = list(colors.keys())
    title_position = {'x': 0.44, 'y': 0.96}

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=['Chum Spawning', 'Coho Spawning', 'Chum Female Spawning', 'Chum Male Spawning'],
        shared_xaxes=True,
        vertical_spacing=0.12
    )

    legend_shown = {f'{c}-{year}': False for c in categories for year in years}
    traces = []

    for year in years:
        for species, sex, row, col in species_configs:
            sex_filter = f"AND Sex = '{sex}'" if sex else ''
            query = f"""
                SELECT
                    COUNT(CASE WHEN Spawned = 'Spawned' THEN _id END) AS Spawned,
                    COUNT(CASE WHEN Spawned = 'Unspawned' THEN _id END) AS Unspawned,
                    COUNT(CASE WHEN Spawned IN ('Partially_spawned', 'Partially spawned') THEN _id END) AS "Partially Spawned",
                    COUNT(CASE WHEN Spawned = 'Unknown' THEN _id END) AS Unknown
                FROM salmon
                WHERE year = {year} AND Species = '{species}' AND Type = 'Dead' {sex_filter}
            """
            df = pd.read_sql(query, connection)
            if df.empty:
                counts = {c: 0 for c in categories}
            else:
                counts = {c: int(df.at[0, c]) if c in df.columns else 0 for c in categories}
            total = sum(counts.values())

            label = species if not sex else f"{species} {sex}"

            for category in categories:
                count = counts.get(category, 0)
                pct = 0 if total == 0 else 100.0 * count / total

                showlegend = False
                legend_key = f'{category}-{year}'
                if not legend_shown[legend_key] and row == 1 and col == 1:
                    showlegend = True
                    legend_shown[legend_key] = True

                tr = go.Bar(
                    x=[pct],
                    y=[label],
                    orientation='h',
                    name=category,
                    marker_color=colors[category],
                    visible=(year == default_year),
                    text=[f'{pct:.0f}%'],
                    textposition='inside',
                    customdata=[[count]],
                    hovertemplate=category + ': %{customdata[0]}<br>Percent: %{x:.1f}%<extra></extra>',
                    legendgroup=category,
                    showlegend=showlegend
                )
                fig.add_trace(tr, row=row, col=col)
                traces.append(tr)

    buttons = []
    traces_per_year = len(traces) // len(years)
    for i, year in enumerate(years):
        visible = [False] * len(traces)
        start = i * traces_per_year
        for j in range(start, start + traces_per_year):
            visible[j] = True
        buttons.append(dict(
            label=year,
            method='update',
            args=[
                {'visible': visible},
                {'title': {'text': f'Spawning Success {year}', **title_position}}
            ]
        ))

    fig.update_layout(
        barmode='stack',
        title={'text': f'Spawning Success {default_year}', **title_position},
        font=plotly_font_family,
        updatemenus=[{**dict(
            active=years.index(default_year),
            buttons=buttons,
            x=0, y=1.15,
        ), **plotly_year_menu_attrs}],
        legend_title_text='Status',
        height=620,
        margin=dict(t=120, b=20, l=20, r=20)
    )

    fig.update_xaxes(range=[0, 100], showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    return fig.to_html(include_plotlyjs='cdn', full_html=False)

if use_v2_report:
    spawningChartsByYear = displaySpawningChartsByYear()
    display(HTML(spawningChartsByYear))
else:
    chumSpawnSuccess = plotSpawning('Chum')
    chumMaleSpawnSuccess = plotSpawning('Chum', 'Male')
    chumFemaleSpawnSuccess = plotSpawning('Chum', 'Female')
    cohoSpawnSuccess = plotSpawning('Coho')



In [ ]:
### PREDATION
def plotPredation(species):
    colors = {'Eye loss only': 'teal', 'No damage':'pink', 'Unknown':'grey', 'Predation':'orange'}
    query = f'''
    SELECT
        100 * CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' AND Predation = 'Eye_loss_only' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' THEN _id END) AS float) AS [Eye loss only],
        100 * CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' AND Predation = 'Predation' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' THEN _id END) AS float) AS Predation,
        100 * CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' AND Predation = 'No_damage' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' THEN _id END) AS float) AS [No damage],
        100 * CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' AND Predation = 'Unknown' THEN _id END) AS float) / CAST(COUNT(CASE WHEN Species = '{species}' AND Type = 'Dead' THEN _id END) AS float) AS Unknown
    FROM
        salmon
    WHERE year = {surveyYear}
    '''
    return plotBarH(query, f'{species} Predation', colors)

def displayPredationChartsByYear(years=list(salmonURIs.keys()), default_year=surveyYear):
    species_configs = [('Chum', 1, 1), ('Coho', 1, 2)]
    colors = {
        'Eye loss only': 'teal',
        'Predation': 'orange',
        'No damage': 'pink',
        'Unknown': 'lightgray'
    }
    categories = list(colors.keys())
    title_position = {'x': 0.44, 'y': 0.96}

    fig = make_subplots(rows=1, cols=2, subplot_titles=['Chum Predation', 'Coho Predation'], shared_xaxes=True)

    legend_shown = {f'{c}-{year}': False for c in categories for year in years}
    traces = []

    for year in years:
        for species, row, col in species_configs:
            query = f"""
                SELECT
                    COUNT(CASE WHEN Predation in ('Eye_loss_only', 'Eye loss') THEN _id END) AS "Eye loss only",
                    COUNT(CASE WHEN Predation in ('Yes', 'Predation') THEN _id END) AS Predation,
                    COUNT(CASE WHEN Predation in ('No', 'No_damage') THEN _id END) AS "No damage",
                    COUNT(CASE WHEN Predation in ('', 'Unknown') THEN _id END) AS Unknown
                FROM salmon
                WHERE year = {year} AND Species = '{species}' AND Type = 'Dead'
            """
            df = pd.read_sql(query, connection)
            if df.empty:
                counts = {c: 0 for c in categories}
            else:
                counts = {c: int(df.at[0, c]) if c in df.columns else 0 for c in categories}
            total = sum(counts.values())

            label = species
            for category in categories:
                count = counts.get(category, 0)
                pct = 0 if total == 0 else 100.0 * count / total

                showlegend = False
                legend_key = f'{category}-{year}'
                if not legend_shown[legend_key] and row == 1 and col == 1:
                    showlegend = True
                    legend_shown[legend_key] = True

                tr = go.Bar(
                    x=[pct],
                    y=[label],
                    orientation='h',
                    name=category,
                    marker_color=colors[category],
                    visible=(year == default_year),
                    text=[f'{pct:.0f}%'],
                    textposition='inside',
                    customdata=[[count]],
                    hovertemplate=category + ': %{customdata[0]}<br>Percent: %{x:.1f}%<extra></extra>',
                    legendgroup=category,
                    showlegend=showlegend
                )
                fig.add_trace(tr, row=row, col=col)
                traces.append(tr)

    traces_per_year = len(traces) // len(years)
    buttons = []
    for i, year in enumerate(years):
        visible = [False] * len(traces)
        start = i * traces_per_year
        for j in range(start, start + traces_per_year):
            visible[j] = True
        buttons.append(dict(
            label=year,
            method='update',
            args=[
                {'visible': visible},
                {'title': {'text': f'Predation {year}', **title_position}}
            ]
        ))

    fig.update_layout(barmode='stack',
                      title={'text': f'Predation {default_year}', **title_position},
                      font=plotly_font_family,
                      updatemenus=[{**dict(
                          active=years.index(default_year), buttons=buttons, x=0, y=1.3),
                          **plotly_year_menu_attrs}],
                      legend_title_text='Predation',
                      height=420,
                      margin=dict(t=120, b=20, l=20, r=20))

    fig.update_xaxes(range=[0, 100], showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    return fig.to_html(include_plotlyjs='cdn', full_html=False)

if use_v2_report:
    predationChartsByYear = displayPredationChartsByYear()
    display(HTML(predationChartsByYear))
else:
    chumPredation = plotPredation('Chum')
    cohoPredation = plotPredation('Coho')

In [ ]:
# ## USER INPUT QUERY
# done = False
# while not done:
#     try:
#         query = input("Enter a query: ")
#         print("entering query: " + query)
#         cursor.execute(query)
#         print(cursor.fetchall())
#     except sqlite3.Error as e:
#         print("SQLite error:", e)

In [ ]:
import unittest
class TestNotebook(unittest.TestCase):
    def testYearlyTotals(self):
        actual = getSurveyStats('2021').tail(1)
        # compare 2021 yearly totals with expected values
        self.assertEqual(actual['running_total_all_salmon'].item(), 1008)
        self.assertEqual(actual['running_total_all_chum'].item(), 939)
        self.assertEqual(actual['running_total_all_coho'].item(), 66)
        self.assertEqual(actual['Survey_Date'].item(), '2021-12-07')
    def testSurveyStats(self):
        # compare 2021-11-16 against expected
        actual = getSurveyStats('2021').query('`Survey_Date` == "2021-11-16"')
        self.assertEqual(actual['dead_chum_count'].item(), 114)
        self.assertEqual(actual['dead_coho_count'].item(), 29)
        self.assertEqual(actual['live_chum_count'].item(), 447)
        self.assertEqual(actual['live_coho_count'].item(), 2)
        self.assertEqual(actual['live_cutthroat_count'].item(), 2)
        self.assertEqual(actual['redd_count'].item(), 39)
        self.assertEqual(actual['total_dead_salmon_count'].item(), 143)
        self.assertEqual(actual['total_live_salmon_count'].item(), 451)
        self.assertEqual(actual['running_total_dead_salmon'].item(), 277)
        self.assertEqual(actual['running_total_dead_chum'].item(), 222)
        self.assertEqual(actual['running_total_dead_coho'].item(), 52)
if not aboveDamOnly: unittest.main(argv=[''], exit=False)

In [ ]:
from datetime import date
import plotly.graph_objects as go

def plotSeries(year):
    statsDf = getSurveyStats(year)
    statsDf['Survey_Date'] = statsDf['Survey_Date'].apply(lambda x: datetime.strptime(date.fromisoformat(x).strftime("%m-%d"),"%m-%d"))
    plt.plot('Survey_Date', 'total_salmon_count', data=statsDf, label=year)

def getYearByYearCountPlot():
    fig, ax = plt.subplots()
    for year in salmonURIs:
        plotSeries(year)
    plt.title('Count by time of year')
    plt.ylabel('Count')
    plt.xlabel('Survey Date')
    plt.xticks(rotation = 45)
    plt.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    return getFigureAsHTML()

def getInteractiveYearByYearCountPlot(years=list(salmonURIs.keys())):
    traces = []
    ycol = 'total_salmon_count'
    for year in years:
        df = getSurveyStats(str(year))
        if df.empty:
            continue
        df['Survey_Date_dt'] = pd.to_datetime(df['Survey_Date'])
        df['plot_date'] = pd.to_datetime(df['Survey_Date_dt'].dt.strftime('2000-%m-%d'))
        traces.append(go.Scatter(
            x=df['plot_date'],
            y=df[ycol],
            mode='lines+markers',
            name=str(year),
            customdata=df.assign(year=year)[['year', 'Survey_Date']].values,
            hovertemplate='Year: %{customdata[0]}<br>Date: %{customdata[1]}<br>Count: %{y}<extra></extra>'
        ))

    if not traces:
        return ''

    fig = go.Figure(data=traces)
    fig.update_layout(
        title={'text': 'Count by time of year', 'x': 0.5, 'y': 0.96},
        font=plotly_font_family,
        xaxis=dict(title='Survey date (month-day)', tickformat='%b %d'),
        yaxis=dict(title='Count'),
        legend_title_text='Year',
        hovermode='closest',
        height=480,
        margin=dict(t=50, b=20, l=60, r=20)
    )
    return fig.to_html(include_plotlyjs='cdn', full_html=False)

if use_v2_report:
    yearByYearCountPlot = getInteractiveYearByYearCountPlot()
    display(HTML(yearByYearCountPlot))
else:
    yearByYearCountPlot = getYearByYearCountPlot()

In [ ]:
## ALL SURVEYS SCATTER MAP
YEAR_SCATTER_MAP_QUERY = f'''
SELECT
    Survey_Date, Type, Species, Latitude, Longitude, Accuracy, Distance, Sex, Quantity
FROM
    salmon
WHERE Latitude IS NOT NULL AND Accuracy < 50 AND year = ? {additionalFilterForAboveDam}
'''

def displayScatterMapByYear(years=list(salmonURIs.keys()), default_year=surveyYear):
    traces = []
    color_dict = {'Live': 'teal', 'Redd': 'red', 'Dead': 'black'}
    types = list(color_dict.keys())
    title_position = {'x': 0.5, 'y': 0.96}

    for year in years:
        df = pd.read_sql(YEAR_SCATTER_MAP_QUERY, con=connection, params=[year])
        df['marker_color'] = df['Type'].map(color_dict)
        for t in types:
            color = color_dict[t]
            subset = df[df['Type'] == t]
            traces.append(go.Scattermapbox(
                lat=subset['Latitude'],
                lon=subset['Longitude'],
                mode='markers',
                name=t,
                legendgroup=t,
                marker=dict(color=color, size=8),
                text=subset['Type'],
                customdata=subset[['Distance','Quantity','Species','Sex','Accuracy']].values,
                hovertemplate=(
                    '<b>Type: %{text}</b><br>'
                    'Distance: %{customdata[0]}<br>'
                    'Quantity: %{customdata[1]}<br>'
                    'Species: %{customdata[2]}<br>'
                    'Sex: %{customdata[3]}<br>'
                    'Accuracy: %{customdata[4]}<extra></extra>'
                ),
                visible=(year == default_year),
                showlegend=(year == default_year)
            ))

    buttons = []
    for year in years:
        visible = [(y == year) for y in years]
        buttons.append(dict(
            label=year,
            method='update',
            args=[
                {'visible': visible, 'showlegend': True},
                {'title': {'text': f'{year} Fish Scatter Map', **title_position}}
            ]
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title={'text': f'{default_year} Fish Scatter Map', **title_position},
        font=plotly_font_family,
        mapbox_style='open-street-map',
        mapbox_center={'lat': 47.71157, 'lon': -122.3759},
        mapbox_zoom=15,
        updatemenus=[{**dict(
            active=years.index(default_year),
            buttons=buttons,
            pad={'r': 10, 't': 10},
            x=0,
            y=1.15,
        ), **plotly_year_menu_attrs}],
        legend=dict(
            orientation='v',
            x=1.02,
            y=0.9,
            xanchor='left',
            yanchor='middle',
            traceorder='normal'
        ),
        legend_title_text='Type',
        margin=dict(t=100, b=20, l=20, r=20)
    )

    return fig.to_html(include_plotlyjs='cdn', full_html=False)

if use_v2_report:
    scatterMapByYear = displayScatterMapByYear()
    display(HTML(scatterMapByYear))
else:
    df = pd.read_sql(YEAR_SCATTER_MAP_QUERY, con=connection, params=[surveyYear])
    yearScatterMap = getScatterMap(df, f'{surveyYear} Fish Scatter Map')

In [ ]:
##LATEST SURVEY SCATTER MAP
LATEST_YEAR_SCATTER_MAP_QUERY = f'''
SELECT
    Survey_Date, Type, Species, Latitude, Longitude, Accuracy, Distance, Sex, Quantity
FROM
    salmon
WHERE Latitude IS NOT NULL AND Accuracy < 50 AND year = ? AND Survey_Date = (SELECT MAX(Survey_Date) FROM salmon WHERE year = ?) {additionalFilterForAboveDam}
'''
def displayLatestScatterMap(year=surveyYear):
    traces = []
    color_dict = {'Live': 'teal', 'Redd': 'red', 'Dead': 'black'}
    types = list(color_dict.keys())
    df = pd.read_sql(LATEST_YEAR_SCATTER_MAP_QUERY, con=connection, params=[year, year])
    latestSurvey = df.iloc[0]['Survey_Date']

    df['marker_color'] = df['Type'].map(color_dict)
    for t in types:
        color = color_dict[t]
        subset = df[df['Type'] == t]
        traces.append(go.Scattermapbox(
            lat=subset['Latitude'],
            lon=subset['Longitude'],
            mode='markers',
            name=t,
            legendgroup=t,
            marker=dict(color=color, size=8),
            text=subset['Type'],
            customdata=subset[['Distance','Quantity','Species','Sex','Accuracy']].values,
            hovertemplate=(
                '<b>Type: %{text}</b><br>'
                'Distance: %{customdata[0]}<br>'
                'Quantity: %{customdata[1]}<br>'
                'Species: %{customdata[2]}<br>'
                'Sex: %{customdata[3]}<br>'
                'Accuracy: %{customdata[4]}<extra></extra>'
            ),
            visible=True,
            showlegend=True
        ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title={'text': f'{latestSurvey} Fish Scatter Map', 'x': 0.5, 'y': 0.96},
        font=plotly_font_family,
        mapbox_style='open-street-map',
        mapbox_center={'lat': 47.71157, 'lon': -122.3759},
        mapbox_zoom=15,
        legend=dict(
            orientation='v',
            x=1.02,
            y=0.9,
            xanchor='left',
            yanchor='middle',
            traceorder='normal'
        ),
        legend_title_text='Type',
        margin=dict(t=50, b=20, l=20, r=20),
    )

    return fig.to_html(include_plotlyjs='cdn', full_html=False)

if use_v2_report:
    latestScatterMap = displayLatestScatterMap()
    display(HTML(latestScatterMap))
else:
    df = pd.read_sql(LATEST_YEAR_SCATTER_MAP_QUERY, con=connection, params=[surveyYear, surveyYear])
    latestSurvey = df.iloc[0]['Survey_Date']
    latestScatterMap = getScatterMap(df, f'{latestSurvey} Fish Scatter Map')

In [ ]:
## GENERATE REPORT WITH WHICHEVER FIGURES WERE CREATED
from jinja2 import Environment, FileSystemLoader
currentTimePacific = datetime.now(pytz.timezone('America/Los_Angeles')).strftime('%Y-%m-%d_%H-%M-%S') #cannot use system time due to colab
reportFileName = currentTimePacific + '_salmonReport.html'
def generateReport():
    if IN_COLAB:
        templatePath = 'Survey-Notebook/templates'
    else:
        templatePath = 'templates'
    env = Environment(loader=FileSystemLoader(templatePath))
    template = env.get_template(f"report_template{'_v2' if use_v2_report else ''}.html")
    reportData = {}
    reportData['reportGenTime'] = currentTimePacific
    reportData['reddsTable'] = reddsTable
    reportData['yearByYearCountPlot'] = yearByYearCountPlot
    reportData['countPlot'] = countPlot
    reportData['surveyStatsTable'] = surveyStatsTable
    reportData['yearScatterMap'] = yearScatterMap
    reportData['latestScatterMap'] = latestScatterMap
    reportData['maxSurveyChum'] = maxSurveyChum
    reportData['maxSurveyChumDate'] = maxSurveyChumDate
    reportData['maxSurveyCoho'] = maxSurveyCoho
    reportData['maxSurveyCohoDate'] = maxSurveyCohoDate
    reportData['chumSpawnSuccess'] = chumSpawnSuccess
    reportData['cohoSpawnSuccess'] = cohoSpawnSuccess
    reportData['chumFemaleSpawnSuccess'] = chumFemaleSpawnSuccess
    reportData['chumMaleSpawnSuccess'] = chumMaleSpawnSuccess
    reportData['chumPredation'] = chumPredation
    reportData['cohoPredation'] = cohoPredation
    reportData['countPlotByYear'] = countPlotByYear
    reportData['spawningChartsByYear'] = spawningChartsByYear
    reportData['predationChartsByYear'] = predationChartsByYear
    reportData['scatterMapByYear'] = scatterMapByYear
    html = template.render(reportData)
    with open(reportFileName, 'w') as f:
        f.write(html)
generateReport()

In [ ]:
#Download HTML report if in google colab
if IN_COLAB:
    from google.colab import files
    files.download(reportFileName)